In [50]:
#!pip install xlrd

In [3]:
# import libraries
import csv
import json

In [4]:
from google.colab import files

uploaded = files.upload()

Saving appointment.csv to appointment.csv


In [5]:
# part 1: python and data foundation
# task 1.1 :read file
# read csv
with open("appointment.csv", "r",
          newline="", encoding="utf-8") as file:

    reader = csv.DictReader(file)

    print("Header:")
    print(reader.fieldnames)

    rows = list(reader)

    print("\nTotal number of rows:", len(rows))

Header:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']

Total number of rows: 1020


In [6]:
# parse_rows() function
def parse_row(row):
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row



In [7]:
# separate accepted and rejected records

accepted = []
rejected = []

for row in rows:
    try:
        converted_row = parse_row(row.copy())
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))


Accepted rows: 988
Rejected rows: 32


In [8]:
# save rejected file into json

with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Rejected rows saved successfully!")


Rejected rows saved successfully!


In [9]:
# TASK 1.2: inspect and clean data
#import libraries

import pandas as pd
import numpy as np

# load dataset

df = pd.read_csv("appointment.csv")

# remove duplicate
df = df.drop_duplicates()
print("Rows after removing duplicates:", len(df))


Rows after removing duplicates: 1000


In [10]:
#clean booking channel
df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

print(df["booking_channel"].value_counts())
print("Unique booking channels:",
      df["booking_channel"].nunique())


booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64
Unique booking channels: 3


In [11]:
#convert date columns
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"]
)


In [12]:
# remove rows with missing appointment date or status

df = df.dropna(subset=["appt_date", "status"])

print("Rows after removing missing dates and status:",
      len(df))


Rows after removing missing dates and status: 990


In [13]:
# TASK 1.3: Clean validity, missing values, outliers
# identify invalid age
df.loc[
    (df["age"] < 1) | (df["age"] > 100),
    "age"
] = np.nan

print("Missing ages after validity check:",
      df["age"].isnull().sum())


Missing ages after validity check: 17


In [14]:
# fill missing values median
age_median = df["age"].median()
distance_median = df["distance_km"].median()

print("Age median:", age_median)
print("Distance median:", distance_median)

df["age"] = df["age"].fillna(age_median).round().astype(int)

df["distance_km"] = df["distance_km"].fillna(
    distance_median
)

print("\nMissing values after filling:")
print(df[["age", "distance_km"]].isnull().sum())


Age median: 48.0
Distance median: 8.25

Missing values after filling:
age            0
distance_km    0
dtype: int64


In [15]:
# handle distance outliers using IQR
Q1 = df["distance_km"].quantile(0.25)
Q3 = df["distance_km"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

distance_capped = (
    (df["distance_km"] < lower) |
    (df["distance_km"] > upper)
).sum()

df["distance_km"] = np.clip(
    df["distance_km"], lower, upper
)

print("Distance Q1:", Q1)
print("Distance Q3:", Q3)
print("Distance IQR:", IQR)
print("Distance lower fence:", lower)
print("Distance upper fence:", upper)
print("Distance values capped:", distance_capped)


Distance Q1: 4.625
Distance Q3: 13.0
Distance IQR: 8.375
Distance lower fence: -7.9375
Distance upper fence: 25.5625
Distance values capped: 26


In [16]:
# handle fee outliers

Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

fee_capped = (
    (df["fee_inr"] < lower_fee) |
    (df["fee_inr"] > upper_fee)
).sum()

df["fee_inr"] = np.clip(
    df["fee_inr"], lower_fee, upper_fee
)

print("Fee Q1:", Q1_fee)
print("Fee Q3:", Q3_fee)
print("Fee IQR:", IQR_fee)
print("Fee lower fence:", lower_fee)
print("Fee upper fence:", upper_fee)
print("Fee values capped:", fee_capped)


Fee Q1: 500.0
Fee Q3: 787.5
Fee IQR: 287.5
Fee lower fence: 68.75
Fee upper fence: 1218.75
Fee values capped: 3


In [17]:
# round the values

df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)

print(df[["age", "distance_km", "fee_inr"]].head())


   age  distance_km  fee_inr
0   60         18.0      850
1   39         12.5      350
2   63          4.1      850
4   52          7.5      750
5   64         19.0      700


In [18]:
#TASK 1.4: Feature engineering and numpy summary
# create lead_day

df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

print(df["lead_days"].describe())


count    990.000000
mean      11.819192
std        8.738108
min        1.000000
25%        5.000000
50%       10.000000
75%       16.000000
max       45.000000
Name: lead_days, dtype: float64


In [19]:
# create no_show column

df["no_show"] = np.where(
    df["status"] == "No-Show", 1, 0
)

print(df["no_show"].value_counts())


no_show
0    772
1    218
Name: count, dtype: int64


In [20]:
# convert selected column into numpy
X = df[
    ["age", "distance_km", "fee_inr", "lead_days"]
].to_numpy()

print("NumPy Array:")
print(X.shape)


NumPy Array:
(990, 4)


In [21]:
# calculate column means

column_means = np.mean(X, axis=0)

print("Column means:")
print(column_means)


Column means:
[ 48.17373737   9.50090909 635.71414141  11.81919192]


In [22]:
# find appointment booked more than 14 days ahed
mask = df["lead_days"] > 14

print("Appointments above 14 days:", mask.sum())

print("Percentage:",
      round(mask.mean() * 100, 1), "%")


Appointments above 14 days: 296
Percentage: 29.9 %


In [23]:
# calculate distance z scores
distance = df["distance_km"].to_numpy()

z_scores = (
    distance - np.mean(distance)
) / np.std(distance)

outlier_count = np.sum(np.abs(z_scores) > 2)

print("Distance z-score outliers:", outlier_count)


Distance z-score outliers: 53


In [26]:
from google.colab import files

uploaded = files.upload()

Saving clinic.csv to clinic.csv


In [27]:
#TASK 1.5: Enrich clinic data using Open-Meteo API
# libraries
import requests
import time

# load the clinic dataset

clinic_df = pd.read_csv("clinic.csv")

print(clinic_df.head())
print("Total clinics:", len(clinic_df))



  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873
Total clinics: 8


In [28]:
# fetch temperature
def get_temperature(latitude, longitude, clinic_name):

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    for attempt in range(2):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                raise ValueError(
                    f"Status code: {response.status_code}"
                )

            data = response.json()

            temperature = data["hourly"]["temperature_2m"][0]

            if temperature is None:
                raise ValueError("Temperature is missing")

            return temperature

        except (
            requests.exceptions.RequestException,
            ValueError,
            KeyError,
            IndexError,
            TypeError
        ) as error:

            if attempt == 0:
                print(f"Retrying {clinic_name}: {error}")
                time.sleep(1)
            else:
                print(
                    f"Failed for {clinic_name}: {error}"
                )

    return None


In [29]:
# fetch temperature for all clincs

temperatures = []

for index, row in clinic_df.iterrows():

    temp = get_temperature(
        row["latitude"],
        row["longitude"],
        row["clinic_name"]
    )

    temperatures.append(temp)

clinic_df["temp_c"] = temperatures

print(clinic_df)


Retrying Kothrud Family Clinic: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Failed for Kothrud Family Clinic: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Retrying Hadapsar Care Centre: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Failed for Hadapsar Care Centre: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Retrying Sitabuldi Health Point: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Failed for Sitabuldi Health Point: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Retrying Malviya Nagar Clinic: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Failed for Malviya Nagar Clinic: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Re

In [30]:
# save enriched clinic dataset

clinic_df.to_csv(
    "clinic_enriched.csv",
    index=False
)

print("Clinic data enriched successfully!")


Clinic data enriched successfully!


In [31]:
#TASK 1.6: EXPORT CLEANED DATA AND CREATE SQL FILE
# check appointment id uniquness
print(
    "Unique appointment IDs:",
    df["appointment_id"].nunique()
)

print(
    "Total appointment rows:",
    len(df)
)

Unique appointment IDs: 990
Total appointment rows: 990


In [32]:
# format date column
df["booked_on"] = pd.to_datetime(df["booked_on"])

df["appt_date"] = pd.to_datetime(df["appt_date"])

df["booked_on"] = df["booked_on"].dt.strftime(
    "%Y-%m-%d"
)

df["appt_date"] = df["appt_date"].dt.strftime(
    "%Y-%m-%d"
)


In [33]:
#select exactly 16 columns
columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

df = df[columns]

print("Final shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())


Final shape: (990, 16)

Missing values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64


In [34]:
# save appointment_clean csv
df = df.sort_values("appointment_id")

df.to_csv(
    "appointments_clean.csv",
    index=False
)

print("Clean appointment file saved!")


Clean appointment file saved!


In [35]:
# generate sql insert statement
def sql_value(value, text_value=False):

    if pd.isna(value):
        return "NULL"

    if text_value:
        return "'" + str(value).replace("'", "''") + "'"

    return str(value)


In [56]:
#clinic insert statement
with open("load_data.sql", "w") as file:

    for _, row in clinic_df.iterrows():

        values = [
            sql_value(row["clinic_id"], True),
            sql_value(row["clinic_name"], True),
            sql_value(row["city"], True),
            sql_value(row["tier"], True),
            sql_value(row["latitude"]),
            sql_value(row["longitude"]),
            sql_value(row["temp_c"])
        ]

        statement = (
            "INSERT INTO clinics VALUES ("
            + ",".join(values)
            + ");\n"
        )

        file.write(statement)


In [57]:
# appointment insert statement
text_columns = [
        "appointment_id",
        "patient_id",
        "clinic_id",
        "booked_on",
        "appt_date",
        "gender",
        "booking_channel",
        "visit_type",
        "reminder_sent",
        "status"
    ]

with open("load_data.sql", "a") as file:
    for _, row in df.iterrows():

        values = []

        for column in columns:

            if column in text_columns:
                values.append(
                    sql_value(row[column], True)
                )
            else:
                values.append(
                    sql_value(row[column])
                )

        statement = (
            "INSERT INTO appointment VALUES ("
            + ",".join(values)
            + ");\n"
        )
        file.write(statement)

print("SQL file created successfully!")

SQL file created successfully!


In [58]:
# verify generated files
print("Appointment rows:", len(df))
print("Appointment columns:", len(df.columns))
print("Clinic rows:", len(clinic_df))
print("Unique IDs:", df["appointment_id"].nunique())
print("Missing appointment values:", df.isnull().sum().sum())

with open("load_data.sql", "r") as file:
    statements = file.readlines()

print("Total SQL INSERT statements:", len(statements))


Appointment rows: 990
Appointment columns: 16
Clinic rows: 8
Unique IDs: 990
Missing appointment values: 0
Total SQL INSERT statements: 998


In [61]:
# download load data sql file
from google.colab import files

files.download("load_data.sql")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [62]:
# ignore duplicate record
file_path = "/content/load_data.sql"

with open(file_path, "r", encoding="utf-8") as file:
    sql = file.read()

sql = sql.replace("INSERT INTO", "INSERT IGNORE INTO")

with open("/content/load_data_ignore.sql", "w", encoding="utf-8") as file:
    file.write(sql)

print("All INSERT statements updated successfully!")

All INSERT statements updated successfully!


In [63]:
from google.colab import files

files.download("/content/load_data_ignore.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [66]:
# verify sql result
# Load cleaned appointment data
df = pd.read_csv("appointments_clean.csv")

# Load clinic information
clinic_df = pd.read_csv("clinic_enriched.csv")

# Group appointments clinic-wise
clinic_summary = df.groupby("clinic_id").agg(
    appointment=("appointment_id", "count"),
    no_shows=("no_show", "sum")
).reset_index()

# Calculate percentage
clinic_summary["no_show_pct"] = (
    100 * clinic_summary["no_shows"] /
    clinic_summary["appointment"]
).round(1)

# Add clinic names and cities
clinic_summary = clinic_summary.merge(
    clinic_df[["clinic_id", "clinic_name", "city"]],
    on="clinic_id"
)

# Sort by percentage
clinic_summary = clinic_summary.sort_values(
    "no_show_pct",
    ascending=False
)

print(clinic_summary)

  clinic_id  appointment  no_shows  no_show_pct               clinic_name  \
7       C08           90        30         33.3    Gomti Nagar Care Point   
1       C02          143        39         27.3      Hadapsar Care Centre   
3       C04          142        38         26.8      Malviya Nagar Clinic   
0       C01          160        36         22.5     Kothrud Family Clinic   
6       C07          101        21         20.8        Vijay Nagar Clinic   
5       C06          142        23         16.2  Edappally Medical Centre   
2       C03           82        13         15.9    Sitabuldi Health Point   
4       C05          130        18         13.8     Vaishali Nagar Clinic   

      city  
7  Lucknow  
1     Pune  
3   Jaipur  
0     Pune  
6   Indore  
5    Kochi  
2   Nagpur  
4   Jaipur  
